# Analysis Code for : Leveraging High-Throughput Data to estimate Leukocyte Telomere Length in the Million Veteran Program

**Author:** Prathima Vembu <br>
**Contact:** pvembu@stanford.edu

This repository contains the custom analysis scripts and computational workflow associated with the manuscript published in *PLOS ONE*.

### Data Availability
- **Individual-Level Data:** Individual-level datasets underlying this study cannot be shared outside the Department of Veterans Affairs (VA). 
- **Summary Statistics:** Summary statistics presented in the study are fully available within the manuscript's Results section.

### Code Availability
All custom analysis scripts have been uploaded to this repository and archived via Zenodo(Update link) to ensure reproducibility.

### Repository Structure
- `analysis-notebook.ipynb` - Contains the analysis notebook
- `requirements.txt` - Python package dependencies and version numbers

### Software Environment & Requirements
The analyses were conducted using Python (version 3.10.4)


In [ ]:
# Upgrade pip and install required dependencies
!pip install --upgrade pip
!pip install google google.cloud.storage pandas matplotlib seaborn scikit-learn statsmodels bioinfokit mpmath

# Import core libraries
import os
import re
import io
import warnings
from google.cloud import storage
import matplotlib.pyplot as plt
import mpmath as mp
from patsy import dmatrix
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import chi2
from scipy.stats import pearsonr, spearmanr
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
from decimal import Decimal

# Suppress minor warnings for a clean review output
warnings.filterwarnings('ignore')

# --- Google Cloud Storage Connection Setup ---

# Replace "PROJECT-ID" and "BUCKET-ID" with actual identifiers
client = storage.Client("PROJECT-ID")
bucket = client.bucket("BUCKET-ID")

## 2. Data loading

### 2.1. Process telomere length files from Google cloud storage (GCS)

In [ ]:
# Replace with actual GCS folder path
prefix = 'PATH/TO/TELSEQ-TXT-FILES/'  
blob_list = bucket.list_blobs(prefix=prefix)

# Initialize an empty list to store individual dataframes
dfs = []

print("Starting file processing from GCS bucket...")

# Loop over each blob in the bucket
for blob in blob_list:
    if blob.name.endswith('.txt'):
        try:
            # Download the blob content as a text string
            data = blob.download_as_text(encoding="utf-8")
            lines = data.split('\n')
            
            # Safety check: ensure the file has enough lines to parse
            if len(lines) > 4:
                # Extract column names from the designated line (index 3)
                header_cols = lines[3].split('\t')[3:43]
                
                # Extract data rows (ignoring empty lines)
                row_data = [x.split('\t')[3:43] for x in lines[3:-1] if x.strip()]
                
                if row_data and header_cols:
                    # Create a dataframe for the individual sample
                    df = pd.DataFrame(row_data, columns=header_cols)
                    
                    # Extract the clean sample name from the file path
                    sample_name = blob.name.split('/')[-1].split('.')[0]
                    
                    # Insert the sample identifier column at the front
                    df.insert(0, 'SHIP-ID', sample_name)
                    
                    dfs.append(df)
                    
        except Exception as e:
            # Catch any formatting anomalies in individual files without crashing the whole loop
            print(f"Skipped {blob.name} due to parsing error: {e}")

# Concatenate all individual sample dataframes into a single master dataframe
if dfs:
    master_mvp_ltl_df = pd.concat(dfs, ignore_index=True)
    print(f"Successfully compiled {len(dfs)} files into 'master_mvp_ltl_df'.")
else:
    master_mvp_ltl_df = pd.DataFrame()
    print("No valid files found or processed.")

# Display the top rows of your compiled dataset
master_mvp_ltl_df.head()

### 2.2. Read in MVP Genetically inferred ancestry (GIA) phenotype from GCS

In [ ]:
# Reading in the MVP phenotype dataset
blob = bucket.blob("PATH/TO/MVP-GIA-PHENOTYPE.csv")
data_phenotype = blob.download_as_string().decode('utf-8')
gia_phenotype = pd.DataFrame([x.split(',') for x in data_phenotype.split('\n')]) 
gia_phenotype = gia_phenotype[1:]
gia_phenotype.columns = ['SHIP-ID','Age','Sex','Ethnicity']
gia_phenotype = gia_phenotype.astype({'SHIP-ID':'str', 'Age':'int', 'Sex':'str', 'Ethnicity':'str'})
print(gia_phenotype.head())

### 2.3. Merge Telomere Data with Demographic / GIA Dataset 

In [ ]:
# Ensure LENGTH_ESTIMATE is numeric
master_mvp_ltl_df['LENGTH_ESTIMATE'] = pd.to_numeric(master_mvp_ltl_df['LENGTH_ESTIMATE'], errors='coerce')

# Select relevant columns from your demographic dataframe 
df_demog_subset = gia_phenotype[['SHIP-ID', 'age_blooddraw', 'sex', 'gia']].copy()

# Perform an inner join on SHIP-ID
merged_df = pd.merge(
    df_demog_subset, 
    master_mvp_ltl_df[['SHIP-ID', 'LENGTH_ESTIMATE']], 
    on='SHIP-ID', 
    how='inner'
)

# Drop duplicate rows (frequent when demographic tables have multiple records per ID)
merged_df_unique_dups_removed = merged_df.drop_duplicates().reset_index(drop=True)

print(f"Successfully merged datasets. Total unique records: {len(merged_df_unique_dups_removed)}")
merged_df_unique_dups_removed.head()

### 2.4. Filter cohort for ages 20 to 80 and remove NAs

In [ ]:
# Assuming 'merged_df_unique' is your combined dataset from the previous step
# Rename columns if needed to match your standard schema ('age_blooddraw', 'sex', 'gia', 'LENGTH_ESTIMATE')

# Filter for valid records and age range 20-80
merged_df_unique_dups_removed_age_filtered = merged_df_unique_dups_removed[
    (merged_df_unique_dups_removed['age_blooddraw'] >= 20) & (merged_df_unique_dups_removed['age_blooddraw'] <= 80)
].reset_index(drop=True)

print(f"Total samples after filtering (Age 20-80 & NA removal): {len(merged_df_unique_dups_removed_age_filtered):,}")

### 2.5. Sample counts by Sex and GIA

In [ ]:
print("\n--- Sample Counts by Sex ---")
print(merged_df_unique_dups_removed_age_filtered['sex'].value_counts())

print("--- Sample Counts by GIA ---")
print(merged_df_unique_dups_removed_age_filtered['gia'].value_counts())

### 2.6. Descriptive Statistics 

#### 2.6.1. Descriptive Statistics : Age

In [ ]:
# Calculate summary metrics for Age
age_mean = merged_df_unique_dups_removed_age_filtered['age_blooddraw'].mean()
age_median = merged_df_unique_dups_removed_age_filtered['age_blooddraw'].median()
age_std = merged_df_unique_dups_removed_age_filtered['age_blooddraw'].std()
age_q1 = merged_df_unique_dups_removed_age_filtered['age_blooddraw'].quantile(0.25)
age_q3 = merged_df_unique_dups_removed_age_filtered['age_blooddraw'].quantile(0.75)
age_iqr = age_q3 - age_q1

# Print summary statistics for Age
print("--- Age summary statistics ---")
print(f"Mean: {age_mean:.2f} years")
print(f"Median: {age_median:.2f} years")
print(f"Std Dev: {age_std:.2f} years")
print(f"Q1 (25th percentile): {age_q1:.2f} years")
print(f"Q3 (75th percentile): {age_q3:.2f} years")
print(f"IQR: {age_iqr:.2f} years")

# Plot age distribution 
plt.figure(figsize=(8, 5))
sns.histplot(merged_df_unique_dups_removed_age_filtered['age_blooddraw'], color='skyblue', kde=True, bins=30)
plt.xlabel('Age (years)')
plt.ylabel('Frequency')
plt.title('Figure 1: Distribution of Age in MVP Cohort')
plt.tight_layout()
plt.show()

#### 2.6.1.1. Descriptive Statistics : Age by Sex

In [ ]:
#Violin plot for Age distribution by Sex

# Set figure size
plt.rcParams['figure.figsize'] = [5,5]

# Create the violin plot
ax = sns.violinplot(data=merged_df_unique_dups_removed_age_filtered, y="age_blooddraw", hue="sex", split=True, 
               palette={"M": "steelblue", "F": "darkgoldenrod"})

# Calculate and annotate the number of samples
# Group data by GIA and sex
grouped_data = merged_df_unique_dups_removed_age_filtered.groupby(['sex']).size().reset_index(name='count')

# Create a dictionary to easily access counts by sex
count_dict = dict(zip(grouped_data['sex'], grouped_data['count']))

# Define x-axis positions manually based on the order in the plot
x_positions = {'M': -0.15, 'F': 0.15}  # Adjust these values if needed based on your actual plot

# Annotate the counts on the plot at a fixed y-axis position
for sex, x_pos in x_positions.items():
    count = count_dict.get(sex, 0)
    plt.text(x=x_pos, y=65, s=f'n={count}',  # Fixed y-axis position at 65
             horizontalalignment='center', verticalalignment='top', fontsize=10, color='black', weight='semibold')

# Adjust layout and show the plot
plt.tight_layout()
plt.ylim(0,100)
plt.show()

# Add axis titles and plot title
plt.ylabel('Age (years)')
plt.title('Figure 2: Distribution of "Age", split by Sex')
plt.grid(True, color='gray')

#### 2.6.1.2. Descriptive Statistics : Age by GIA

In [ ]:
#Violin plot for Age distribution by GIA aplit by Sex

# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None

# Function to lighten colors
def lighten_color(color, amount=0.3):
    c = mcolors.to_rgb(color)
    return mcolors.to_hex([min(1, x + amount) for x in c])

# Function to darken colors
def darken_color(color, amount=0.3):
    c = mcolors.to_rgb(color)
    return mcolors.to_hex([max(0, x - amount) for x in c])

# Define colors for each GIA and generate lighter and darker shades
gia_colors = {
    "EUR": "steelblue",
    "AFR": "darkorchid",
    "AMR": "darkorange"
}

palette = {}
for gia, color in gia_colors.items():
    palette[f'{gia}_M'] = darken_color(color, amount=0.06)
    palette[f'{gia}_F'] = lighten_color(color, amount=0.4)

# Create a single figure with subplots
fig, axes = plt.subplots(nrows=1, ncols=len(gia_colors))

# Plotting loop
for ax, gia in zip(axes, gia_colors.keys()):
    subset = merged_df_unique_dups_removed_age_filtered[
        merged_df_unique_dups_removed_age_filtered['gia'] == gia]
    
    # Set figure size
    plt.rcParams['figure.figsize'] = [14, 5]

     # Create a new column for color mapping
    subset.loc[:, 'gia_sex'] = subset['gia'] + '_' + subset['sex']
    
    # Create the violin plot for the subset
    sns.violinplot(
        data=subset,
        x="gia", y="age_blooddraw", hue="gia_sex", split=True,
        palette=palette,
        order=[gia], 
        ax=ax  # Ensure only the current GIA is plotted
    )
    
    # Add axis titles and plot title
    ax.set_xlabel('') 
    ax.set_ylabel('Age (years)')
    ax.set_ylim(0,100)
    # Customize the legend for each subplot
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles, labels,
        title='',
        bbox_to_anchor=(0.74,1.0),  
        loc='upper left',              
        fontsize='8',            
        handlelength=2            
    )

    # Annotate counts for each half of the violin plot
    for sex in ['M', 'F']:
        count = subset[subset['sex'] == sex].shape[0]
        x_offset = -0.2 if sex == 'M' else 0.2  # Adjust position based on sex
        # Place text annotations
        ax.text(
            x=x_offset,  
            y=ax.get_ylim()[1] * 0.53,  # Adjust y position as needed
            s=f'n={count}', 
            horizontalalignment='center', 
            verticalalignment='top', 
            fontsize=10, 
            color='black', 
            weight='semibold'
        )

    ax.grid(True, color='gray')


# Set a main title for the entire figure
fig.suptitle('Figure 3: Distribution of Age, per GIA split by Sex', fontsize=16, y=0.98)  


#### 2.6.2. Descriptive Statistics : LTL

In [ ]:
# Calculate summary metrics for LTL
ltl_mean = merged_df_unique_dups_removed_age_filtered['LENGTH_ESTIMATE'].mean()
ltl_median = merged_df_unique_dups_removed_age_filtered['LENGTH_ESTIMATE'].median()
ltl_std = merged_df_unique_dups_removed_age_filtered['LENGTH_ESTIMATE'].std()
ltl_q1 = merged_df_unique_dups_removed_age_filtered['LENGTH_ESTIMATE'].quantile(0.25)
ltl_q3 = merged_df_unique_dups_removed_age_filtered['LENGTH_ESTIMATE'].quantile(0.75)
ltl_iqr = ltl_q3 - ltl_q1

# Print statistics cleanly for review
print("--- Telomere Length (LTL) Summary Statistics ---")
print(f"Mean: {ltl_mean:.3f} kb")
print(f"Median: {ltl_median:.3f} kb")
print(f"Std Dev: {ltl_std:.3f} kb")
print(f"Q1 (25th percentile): {ltl_q1:.3f} kb")
print(f"Q3 (75th percentile): {ltl_q3:.3f} kb")
print(f"IQR: {ltl_iqr:.3f} kb")

# Plot LTL Distribution
plt.figure(figsize=(8, 5))
sns.histplot(merged_df_unique_dups_removed_age_filtered['LENGTH_ESTIMATE'], color='lightcoral', kde=True, bins=30)
plt.xlabel('Estimated Telomere Length (kilo base pairs)')
plt.ylabel('Frequency')
plt.title('Figure 4: Distribution of Estimated Telomere Length')
plt.tight_layout()
plt.show()

#### 2.6.2.1. Descriptive Statistics : LTL by Sex

In [ ]:
#Violin plot for LTL distribution by Sex

# Set figure size
plt.rcParams['figure.figsize'] = [4,4]

# Create the violin plot
ax = sns.violinplot(data=merged_df_unique_dups_removed_age_filtered, y="LENGTH_ESTIMATE", hue="sex", split=True, 
               palette={"M": "steelblue", "F": "darkgoldenrod"})


# Calculate and annotate the number of samples
# Group data by GIA and sex
grouped_data = merged_df_unique_dups_removed_age_filtered.groupby(['sex']).size().reset_index(name='count')

# Create a dictionary to easily access counts by sex
count_dict = dict(zip(grouped_data['sex'], grouped_data['count']))

# Define x-axis positions manually based on the order in the plot
x_positions = {'M': -0.15, 'F': 0.15}  # Adjust these values if needed based on your actual plot

# Annotate the counts on the plot at a fixed y-axis position
for sex, x_pos in x_positions.items():
    count = count_dict.get(sex, 0)
    plt.text(x=x_pos, y=3.5, s=f'n={count}',  # Fixed y-axis position at 65
             horizontalalignment='center', verticalalignment='top', fontsize=10, color='black', weight='semibold')

# Adjust layout and show the plot
plt.tight_layout()
plt.ylim(0,5)
plt.show()

# Add axis titles and plot title
plt.ylabel('Age (years)')
plt.title('Figure 5: Distribution of estimated LTL, split by Sex', y=1.05)
plt.grid(True, color='gray')

#### 2.6.2.2. Descriptive Statistics : LTL by GIA

In [ ]:
#Violin plot for LTL distribution by GIA split by Sex

# Function to lighten or darken colors
def lighten_color(color, amount=0.3):
    c = mcolors.to_rgb(color)
    return mcolors.to_hex([min(1, x + amount) for x in c])

def darken_color(color, amount=0.3):
    c = mcolors.to_rgb(color)
    return mcolors.to_hex([max(0, x - amount) for x in c])

# Define colors for each GIA and generate lighter and darker shades
gia_colors = {
    "EUR": "steelblue",
    "AFR": "darkorchid",
    "AMR": "darkorange"
}

palette = {}
for gia, color in gia_colors.items():
    palette[f'{gia}_M'] = darken_color(color, amount=0.06)
    palette[f'{gia}_F'] = lighten_color(color, amount=0.4)

# Create a single figure with subplots
fig, axes = plt.subplots(nrows=1, ncols=len(gia_colors))

# Plotting loop
for ax, gia in zip(axes, gia_colors.keys()):
    subset = merged_df_unique_dups_removed_age_filtered[merged_df_unique_dups_removed_age_filtered['gia'] == gia]
    
    # Set figure size
    plt.rcParams['figure.figsize'] = [15, 5]

    # Create a new column for color mapping
    subset['gia_sex'] = subset['gia'] + '_' + subset['sex']
    
    # Create the violin plot for the subset
    sns.violinplot(
        data=subset,
        x="gia", y="LENGTH_ESTIMATE", hue="gia_sex", split=True,
        palette=palette,
        order=[gia], 
        ax=ax  # Ensure only the current GIA is plotted
    )
    
    # Add axis titles and plot title
    ax.set_xlabel('') 
    ax.set_ylabel('Estimated LTL (kbp)')
    ax.set_ylim(0,5)
    # Customize the legend for each subplot
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles, labels,
        title='',
        bbox_to_anchor=(0.74,1.0),  
        loc='upper left',              
        fontsize='8',            
        handlelength=2            
    )

    # Annotate counts for each half of the violin plot
    for sex in ['M', 'F']:
        count = subset[subset['sex'] == sex].shape[0]
        x_offset = -0.2 if sex == 'M' else 0.2  # Adjust position based on sex
        # Place text annotations
        ax.text(
            x=x_offset,  
            y=ax.get_ylim()[1] * 0.45,  # Adjust y position as needed
            s=f'n={count}', 
            horizontalalignment='center', 
            verticalalignment='top', 
            fontsize=10, 
            color='black', 
            weight='semibold' 
        )

    ax.grid(True, color='gray')


# Set a main title for the entire figure
fig.suptitle('Figure 6: Distribution of Estimated LTL per GIA, split by Sex', fontsize=16, y=0.98)  

## 3. Correlation analysis 

### 3.1. Correlation between 'Age' and 'estimated LTL'

In [ ]:
# Define x and y variables for correlation analysis
data = merged_df_unique_dups_removed_age_filtered
x = data['age_blooddraw']
y = data['LENGTH_ESTIMATE']
sample_size = len(data)

# --- 1. Pearson Correlation ---
r_pearson_lt_80, p_pearson_lt_80 = pearsonr(x, y)
p_pearson_decimal_lt_80 = Decimal(str(p_pearson_lt_80))

pearson_matrix_lt_80 = pd.DataFrame(columns=['n', 'pearson-r', 'p-value'])

pearson_matrix_lt_80.loc[''] = [sample_size, round(r_pearson_lt_80, 4), f"{p_pearson_decimal_lt_80:.4e}"]

# --- 2. Spearman Correlation ---
rho_spearman_lt_80, p_spearman_lt_80 = spearmanr(x, y)
p_spearman_decimal_lt_80 = Decimal(str(p_spearman_lt_80))

spearman_matrix_lt_80 = pd.DataFrame(columns=['n', 'spearman-rho', 'p-value'])

spearman_matrix_lt_80.loc[''] = [sample_size, round(rho_spearman_lt_80, 4), f"{p_spearman_decimal_lt_80:.4e}"]

print("Pearson correlation table:")
print(pearson_matrix_lt_80)
print("\nSpearman correlation table:")
print(spearman_matrix_lt_80)

In [ ]:
plt.rcParams['figure.figsize'] = [13,9]

sns.regplot(x=merged_df_unique_dups_removed_age_filtered['age_blooddraw'], 
            y=merged_df_unique_dups_removed_age_filtered['LENGTH_ESTIMATE'], 
            line_kws={'color': 'red'}, 
            scatter_kws={'alpha': 0.5, 'color':'teal'},ci=0.95)
plt.annotate(f"Pearson \n$r^2$: {r_pearson_lt_80:.4f}\(p: {p_pearson_decimal_lt_80:.4e})", xy=(5,4.5),va="top",fontsize=12,bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
plt.annotate(f"Spearman \n$\\rho$: {rho_spearman_lt_80:.4f}(p: {p_spearman_decimal_lt_80:.4e})", xy=(5,3.8),va="top",fontsize=12,bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))

plt.text(85, 4.7, f'n = {len(merged_df_unique_dups_removed_age_filtered)}',
          fontsize=14, verticalalignment='top')
plt.grid(True,color='gray') 
plt.xlim(0,100)
plt.ylim(0)
plt.title(f'Figure 7: Correlation between Age (<=80) and LTL', fontsize=16)
plt.xlabel('Age (years)')
plt.ylabel('Estimated telomere length (kilo base pair)')
plt.show()

### 3.2. Correlation between 'Age' and 'estimated LTL' - stratified by Sex 

In [ ]:
from scipy.stats import pearsonr, spearmanr
from decimal import Decimal

def get_correlations(df):
    x, y = df['age_blooddraw'], df['LENGTH_ESTIMATE']
    
    # Pearson
    r, p_p = pearsonr(x, y)
    # Spearman
    rho, p_s = spearmanr(x, y)
    
    return {
        'count': len(df),
        'r': round(r, 5),
        'p_p': Decimal(str(p_p)),
        'rho': round(rho, 5),
        'p_s': Decimal(str(p_s))
    }

# Define the sex groups and target dataframe
sex_groups = ['M', 'F']
df_main = merged_df_unique_dups_removed_age_filtered

# Calculate metrics for each sex group
sex_stats = {}
for sex in sex_groups:
    subset = df_main[df_main['sex'] == sex]
    sex_stats[sex] = get_correlations(subset)

# --- 1. Pearson Correlation Matrix by Sex ---
pearson_sex_matrix = pd.DataFrame(index=sex_groups, 
                                  columns=['n', 'pearson_r', 'p-value'])

for sex in sex_groups:
    stats = sex_stats[sex]
    pearson_sex_matrix.loc[sex] = [stats['count'], stats['r'], f"{stats['p_p']:.4e}"]

# --- 2. Spearman Correlation Matrix by Sex ---
spearman_sex_matrix = pd.DataFrame(index=sex_groups, 
                                   columns=['n', 'spearman_rho', 'p-value'])

for sex in sex_groups:
    stats = sex_stats[sex]
    spearman_sex_matrix.loc[sex] = [stats['count'], stats['rho'], f"{stats['p_s']:.4e}"]

print("Pearson Correlation by Sex:")
print(pearson_sex_matrix)
print("\nSpearman Correlation by Sex:")
print(spearman_sex_matrix)

In [ ]:
# Setting the size of the plot
plt.rcParams['figure.figsize'] = [12, 7] 

# Define colors
sex_colors = {"M": "steelblue", "F": "darkgoldenrod"}
regression_line_colors = {"M": "steelblue", "F": "peru"}

# Group and sort
sex_groups = merged_df_unique_dups_removed_age_filtered.groupby('sex')
sex_groups = sorted(sex_groups, key=lambda x: len(x[1]), reverse=True)

fig, ax = plt.subplots()
transparency = 0.5  

for sex, group_data in sex_groups:
    color = sex_colors.get(sex, 'gray')
    regression_color = regression_line_colors.get(sex, 'black')

    # Scatter plot
    ax.scatter(group_data['age_blooddraw'], group_data['LENGTH_ESTIMATE'],
               color=color, alpha=transparency, label=f'{sex} - Data')

    # Regression line
    sns.regplot(x=group_data['age_blooddraw'], y=group_data['LENGTH_ESTIMATE'],
                scatter=False, line_kws={'color': regression_color, 'lw': 2.5}, ax=ax)

    # 1. Pearson Calculation
    r_pearson, p_pearson = stats.pearsonr(group_data['age_blooddraw'], group_data['LENGTH_ESTIMATE'])
    
    # 2. Spearman Calculation
    rho_spearman, p_spearman = stats.spearmanr(group_data['age_blooddraw'], group_data['LENGTH_ESTIMATE'])

    # Format p-values using Decimal for scientific notation
    p_p_dec = Decimal(str(p_pearson))
    p_s_dec = Decimal(str(p_spearman))

    # Determine vertical position for the annotation boxes
    y_offset = 4.6 if sex == 'M' else 3.4

    # Combined annotation string (using LaTeX for r^2 and rho)
    # Note: If you want actual r-squared, use r_pearson**2
    ax.annotate(f"{sex}\n"
                f"$r^2$: {r_pearson:.4f} (p: {p_p_dec:.2e})\n"
                f"$\\rho$: {rho_spearman:.4f} (p: {p_s_dec:.2e})",
                xy=(4, y_offset),
                fontsize=11,
                color=color,
                ha='left',
                va='top', 
                bbox=dict(boxstyle="round,pad=0.4", fc="white", ec=color, alpha=0.9))

# Aesthetics
ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Estimated LTL (kbp)', fontsize=12)
ax.set_xlim(0, 100)
ax.set_ylim(0, 5)
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.grid(True, color='lightgray', linestyle='--')

fig.suptitle('Figure 8: Correlation between Age (<= 80) and Estimated LTL - Stratified by Sex', fontsize=14)
plt.text(98, 4.8, f'n = {len(merged_df_unique_dups_removed_age_filtered)}', 
         fontsize=12, verticalalignment='top', horizontalalignment='right')

plt.show()

### 3.2. Correlation between 'Age' and 'estimated LTL' - stratified by GIA 

In [ ]:
def get_correlations(df):
    x, y = df['age_blooddraw'], df['LENGTH_ESTIMATE']
    
    # Pearson
    r, p_p = pearsonr(x, y)
    # Spearman
    rho, p_s = spearmanr(x, y)
    
    return {
        'count': len(df),
        'r': round(r, 5),
        'p_p': Decimal(str(p_p)),
        'rho': round(rho, 5),
        'p_s': Decimal(str(p_s))
    }

# Define the GIA groups
gia_groups = ['EUR', 'AFR', 'AMR']
df_main = merged_df_unique_dups_removed_age_filtered

# Calculate metrics for each GIA group
gia_stats = {}
for gia in gia_groups:
    subset = df_main[df_main['gia'] == gia]
    gia_stats[gia] = get_correlations(subset)

# --- 1. Pearson Correlation Matrix by GIA ---
pearson_gia_matrix = pd.DataFrame(index=gia_groups, 
                                  columns=['n', 'pearson_r', 'p-value'])

for gia in gia_groups:
    stats = gia_stats[gia]
    pearson_gia_matrix.loc[gia] = [stats['count'], stats['r'], f"{stats['p_p']:.4e}"]

# --- 2. Spearman Correlation Matrix by GIA ---
spearman_gia_matrix = pd.DataFrame(index=gia_groups, 
                                   columns=['n', 'spearman_rho', 'p-value'])

for gia in gia_groups:
    stats = gia_stats[gia]
    spearman_gia_matrix.loc[gia] = [stats['count'], stats['rho'], f"{stats['p_s']:.4e}"]

print("Pearson Correlation by GIA:")
print(pearson_gia_matrix)
print("\nSpearman Correlation by GIA:")
print(spearman_gia_matrix)

In [ ]:

# Define the color dictionary for ethnicity
ethnicity_colors = {"EUR": "steelblue", "AFR": "darkorchid", "AMR": "darkorange"}

# Group and sort population groups
population_groups = merged_df_unique_dups_removed_age_filtered.groupby('gia')
population_groups = sorted(population_groups, key=lambda x: len(x[1]), reverse=True)

num_cols = 3
num_rows = 1
fig, axs = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(20, 7)) # Increased height for text

# Ensure axs is iterable even if num_cols is 1
if num_cols == 1:
    axs = [axs]

for i, (population, group_data) in enumerate(population_groups):
    ax = axs[i]
    current_color = ethnicity_colors.get(population, 'gray')

    # Regression Plot
    sns.regplot(x=group_data['age_blooddraw'], y=group_data['LENGTH_ESTIMATE'],
                line_kws={'color': 'red'},
                scatter_kws={'alpha': 0.4, 'color': current_color},
                ax=ax)
    
    # 1. Pearson Calculation
    r_p, p_p = stats.pearsonr(group_data['age_blooddraw'], group_data['LENGTH_ESTIMATE'])
    p_p_dec = Decimal(str(p_p))

    # 2. Spearman Calculation
    rho_s, p_s = stats.spearmanr(group_data['age_blooddraw'], group_data['LENGTH_ESTIMATE'])
    p_s_dec = Decimal(str(p_s))

    # Annotation text with both correlations
    # Using LaTeX for the rho symbol
    stats_text = (f"Pearson \n$r^2$: {r_p:.3f} (p: {p_p_dec:.2e})"
                  f"\nSpearman \n$\\rho$: {rho_s:.3f} (p: {p_s_dec:.2e})")

    # Adding annotation with a white background for readability
    ax.annotate(stats_text, 
                xy=(0.05, 0.8), 
                xycoords='axes fraction',
                va="top", 
                fontsize=11,
                bbox=dict(boxstyle="round,pad=0.3", fc="white",alpha=0.8))

    # Subplot styling
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Estimated LTL (kbp)')
    ax.set_title(f'{population} (n={len(group_data)})')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 5)
    ax.grid(True, color='lightgray', linestyle='--')

# Hide unused subplots
for j in range(len(population_groups), num_cols):
    fig.delaxes(axs[j])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle('Figure 9: Correlation between Age(<=80) and Estimated LTL - Stratified by GIA', fontsize=14)
plt.show()

### 3.3. Correlation between 'Age' and 'estimated LTL' - every 20 years

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

# Ensure the figure size and seaborn style
#plt.rcParams['figure.figsize'] = [10,20]
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Define age groups
bins = [0, 20, 40, 60, 80, float("inf")]
labels = ["<20", "20-40", "40-60", "60-80", ">80"]
merged_df_unique_dups_removed_age_filtered["AgeGroup"] = pd.cut(
    merged_df_unique_dups_removed_age_filtered["age_blooddraw"], bins=bins, labels=labels)

# Define ethnicity groups and colors
ethnicity_groups = ['EUR', 'AFR', 'AMR']

# Define a function to create a colormap with gradients based on colors and age groups
def create_colormap(colors, num_groups):
    label_colors = mcolors.LinearSegmentedColormap.from_list('label_colors', colors)
    return ListedColormap([label_colors(i / num_groups) for i in range(num_groups)])

# Create a figure with subplots for each ethnicity
fig, axes = plt.subplots(nrows=len(ethnicity_groups), figsize=(8,10))

# Set the common x-axis label
fig.text(0.5, -0.02, 'Age (years)', ha='center', fontsize=10)

# Set the common y-axis label
fig.text(-0.02, 0.5, 'Estimated LTL (kbp)', va='center', rotation='vertical', fontsize=10)

# Loop through each ethnicity
for i, ethnicity in enumerate(ethnicity_groups):
    # Filter the data to include only the current ethnicity
    group = merged_df_unique_dups_removed_age_filtered[merged_df_unique_dups_removed_age_filtered["gia"] == ethnicity]

    # Group the data by age group
    grouped = group.groupby("AgeGroup")

    # Get the number of age groups
    num_age_groups = len(grouped)

    # Define the colors for each age bucket within each ethnicity
    if ethnicity == "EUR":
        colors = ["lightsteelblue", "steelblue", "midnightblue"]
    elif ethnicity == "AFR":
        colors = ["plum", "darkorchid", "m"]
    elif ethnicity == "AMR":
        colors = ["peachpuff", "orange", "sienna"]

    # Create a colormap for age groups
    age_group_colors = create_colormap(colors, num_age_groups)

    # Iterate over each age group and create a scatter plot with a regression line
    for age_group, subgroup in grouped:
        if len(subgroup) > 1:  # Check that subgroup has at least two data points
            corr, pval = pearsonr(subgroup["age_blooddraw"], subgroup["LENGTH_ESTIMATE"])
            # Use the age group index to get a color from the colormap
            plot = sns.regplot(x="age_blooddraw", y="LENGTH_ESTIMATE", data=subgroup, 
                               line_kws={"color": "red"}, ax=axes[i], 
                               color=age_group_colors(list(grouped.groups).index(age_group)))
            plot.annotate(f"r: {corr:.2f} \n p: {pval:.02e}", 
                          xy=(subgroup["age_blooddraw"].max(), subgroup["LENGTH_ESTIMATE"].min()-0.3), 
                          ha="right", va="bottom", fontsize=9, weight='bold')
            plot.set_title(f"{ethnicity}", fontsize=10, fontweight='bold')
            plot.set(xlabel="")
            plot.set(ylabel="")
            plot.set_ylim([0, 5])
            plot.set_xlim([0, 100])
            plot.grid(True, color='gray')

            # Get the number of samples and add it as text below the r value
            num_samples = subgroup.shape[0]
            plot.text(subgroup["age_blooddraw"].max(), 4, f"n = {num_samples}", 
                      ha="right", va="bottom", fontsize=9, weight='bold')

# Adjust the layout of the subplots
plt.tight_layout()

# Set the super-title
fig.suptitle('Figure 10 : Correlation between Age (<=80) and estimated LTL, every 20 years', fontsize=10)

# Adjust the layout of the subplots
fig.subplots_adjust(top=0.94)

# Show the plot
plt.show()

## 4. Statistical Testing

In [ ]:
#Male
male_sample = merged_df_unique_dups_removed_age_filtered[
    merged_df_unique_dups_removed_age_filtered['sex'] == 'M']
#Female
female_sample = merged_df_unique_dups_removed_age_filtered[
    merged_df_unique_dups_removed_age_filtered['sex'] == 'F']

#EUR
eur_sample = merged_df_unique_dups_removed_age_filtered[
    merged_df_unique_dups_removed_age_filtered['gia'] == 'EUR']

#AFR
afr_sample = merged_df_unique_dups_removed_age_filtered[
    merged_df_unique_dups_removed_age_filtered['gia'] == 'AFR']

#AMR
amr_sample = merged_df_unique_dups_removed_age_filtered[
    merged_df_unique_dups_removed_age_filtered['gia'] == 'AMR']

### 4.1. Parametric test : t-test

In [ ]:
# Perform an independent t-test
t_statistic, p_value = stats.ttest_ind(male_sample['LENGTH_ESTIMATE'], female_sample['LENGTH_ESTIMATE'])

# Print the results
print(f"t Statistic: {round(t_statistic,4)}")
print(f"P-value: {p_value}")

# Define your significance level (alpha)
alpha = 0.05

# Check if the p-value is less than alpha to determine statistical significance
if p_value < alpha:
    print("Reject null hypothesis. There is a statistically significant difference between Male and Female estimated LTL.")
else:
    print("Fail to reject the null hypothesis. There is no statistically significant difference between Male and Female estimated LTL.")

### 4.2. Parametric test : ANOVA

In [ ]:
#Ordinary Least Squares (OLS) model. The ols function is used to create a model 
# with length_estimate as the dependent variable and GIA as the 
# independent variable. The fit method is used to fit the model to the data
model_merged = ols('LENGTH_ESTIMATE ~ gia', data=merged_df_unique_dups_removed_age_filtered).fit()

#The 'sm.stats.anova_lm' function is used to perform the one-way ANOVA 
# and produce an ANOVA table. The 'typ=2' argument specifies that we 
# want to use the Type 2 sum of squares method
anova_table_merged = sm.stats.anova_lm(model_merged, typ=2)

#Printing the anova table
anova_table_merged

### 4.3. Non-Parametric test : Mann Whitney U test

In [ ]:
# Perform the Mann-Whitney U test
mw_statistic, mw_p_value = mannwhitneyu(male_sample['LENGTH_ESTIMATE'], female_sample['LENGTH_ESTIMATE'])

# Print the results
print(f"Mann-Whitney U Statistic: {mw_statistic}")
print(f"P-value: {mw_p_value}")

# Interpret the results
alpha = 0.05  # significance level
if p_value < alpha:
    print("Reject the null hypothesis. There is a statistically significant difference between Male and Female estimated LTL.")
else:
    print("Fail to reject the null hypothesis. There is no statistically significant difference between Male and Female estimated LTL.")

### 4.4. Non-Parametric test : Kruskal Wallis Test

In [ ]:
# Perform the Kruskal-Wallis test
kw_statistic, kw_p_value = kruskal(eur_sample['LENGTH_ESTIMATE'], afr_sample['LENGTH_ESTIMATE'], amr_sample['LENGTH_ESTIMATE'])

# Print the results
print(f"Kruskal-Wallis Statistic: {kw_statistic}")
print(f"P-value: {kw_p_value}")

# Interpret the results
alpha = 0.05  # significance level
if p_value < alpha:
    print("Reject the null hypothesis. There is a statistically significant difference in telomere length among genetically inferred ancestries.")
else:
    print("Fail to reject the null hypothesis. There is no statistically significant difference in telomere length among genetically inferred ancestries.")

## 5. Per sample TelSeq runtime estimation

In [ ]:
def convert_seconds_to_hours(seconds):
    return seconds / 3600.0

# Specify your GCS folder prefix 
prefix = 'PATH/TO/LOGS/'

# list_blobs automatically searches recursively through all subdirectories under the prefix
blob_list = bucket.list_blobs(prefix=prefix)

data = []

print("Processing log files from GCS...")

for blob in blob_list:
    # Exclude stderr.log and stdout.log files
    if blob.name.endswith(".log") and not (blob.name.endswith("-stderr.log") or blob.name.endswith("-stdout.log")):
        try:
            blob_content = blob.download_as_text()

            # Extract wall clock time using regular expression
            wall_clock_match = re.search(r"wall clock: ([\d.]+)s", blob_content)
            if wall_clock_match:
                wall_clock_seconds = float(wall_clock_match.group(1))
                wall_clock_hours = convert_seconds_to_hours(wall_clock_seconds)
                
                # Extract ID number from CRAM object path 
                id_text_match = re.search(r"([^/]+)\.cram", blob_content)
                if id_text_match:
                    id_text = id_text_match.group(1)
                    data.append((id_text, wall_clock_seconds, wall_clock_hours))
                else:
                    print(f"ID number not found in file path structure: {blob.name}")
            else:
                print(f"Wall clock time not found in file: {blob.name}")
                
        except Exception as e:
            print(f"Error processing {blob.name}: {e}")

# Create a DataFrame from the collected log data
df_logs = pd.DataFrame(data, columns=["SHIP-ID", "Runtime (s)", "Runtime (hours)"])

print(f"Successfully processed {len(df_logs):,} log files.")
df_logs.head()

In [ ]:
# Merge the runtime data with the main LTL dataset based on 'SHIP-ID'
merged_df_unique_dups_removed_age_filtered_with_runtime = pd.merge(
    merged_df_unique_dups_removed_age_filtered,
    df_logs_formatted[['SHIP-ID', 'Runtime (s)', 'Runtime (hours)']],
    on='SHIP-ID',
    how='left'
)

# Verification check
print(f"Original cohort rows: {len(merged_df_unique_dups_removed_age_filtered):,}")
print(f"Rows after merging runtime data: {len(merged_df_unique_dups_removed_age_filtered_with_runtime):,}")
print(f"Number of samples with recorded runtimes: {merged_df_unique_dups_removed_age_filtered_with_runtime['Runtime (hours)'].notna().sum():,}")

merged_df_unique_dups_removed_age_filtered_with_runtime.head()

In [ ]:
#Plot Runtime distribution for TelSeq on MVP WGS genomes
values = merged_df_unique_dups_removed_age_filtered_with_runtime['Runtime (hours)']

# Set Seaborn style
sns.set(style='whitegrid')

# Create a histogram
plt.figure(figsize=(10, 6))
hist_values, bin_edges, _ = plt.hist(values, bins=20, range=(0, max(values)), color='indigo', edgecolor='black', alpha=0.5, histtype='barstacked')
plt.ylim(0, 45000)
plt.xlim(0,5)

# Add labels and title
plt.xlabel('Runtime (hours)')
plt.ylabel('Frequency')
#plt.text(4.2, 42500, f'n = {len(df_logs)}', fontsize=14, verticalalignment='top')
plt.title('Figure 11: Histogram showing per sample runtime for TelSeq '+str(len(df_logs))+' on MVP WGS genomes')

# Annotate each bar with the count of samples
for i, count in enumerate(hist_values):
    if count > 0:
     plt.text(
        bin_edges[i] + 0.12,
        count + 1250,
        f'{int(count)}',
        fontsize=9,
        verticalalignment='top',
        horizontalalignment='center', 
        #weight='bold'
    )

# Show the plot
plt.show()

### 6.1. Model 1 - Telomere length ~ age + sex

In [ ]:
model_1 = sm.OLS.from_formula(formula='LENGTH_ESTIMATE ~ age_blooddraw + sex', 
                              data=merged_df_unique_dups_removed_age_filtered_with_runtime).fit()
results_1 = model_1.summary()
print(results_1)

### 6.2. Model 2 - Telomere length ~ age + sex + ancestry

In [ ]:
# Specify 'gia' as categorical and set 'gia_EUR' as the reference category
model_2 = sm.OLS.from_formula(formula='LENGTH_ESTIMATE ~ age_blooddraw + sex + C(gia, Treatment("EUR"))',
                              data=merged_df_unique_dups_removed_age_filtered_with_runtime).fit()
results_2 = model_2.summary()
print(results_2)

### 6.3. Model 3 - LTL ~ age + sex (each ancestry)

In [ ]:
# List of gia groups
gia_groups = ["EUR", "AFR", "AMR"]

# Define the formula for multiple regression
formula = 'LENGTH_ESTIMATE ~ age_blooddraw + sex'

# Loop through each ancestry group
for gia in gia_groups:
    # Subset the data for the specific ancestry
    subset_data = merged_df_unique_dups_removed_age_filtered_with_runtime[
        merged_df_unique_dups_removed_age_filtered_with_runtime["gia"] == gia]

    # Perform multiple regression for the current ancestry group
    model3 = sm.OLS.from_formula(formula, data=subset_data).fit()

    # Get model summary
    model3_summary = model3.summary()

    # Print the results for the current ancestry group
    print(f"GIA: {gia}")
    print(model3_summary)

In [ ]:
# Regression plots for Telomere length ~ age + sex (each GIA)

# List of gia groups
gia_groups = ["EUR", "AFR", "AMR"]
#List of colours
colors = ["steelblue", "darkorchid","darkorange"]

# Set up the figure and axes
fig, axes = plt.subplots(nrows=1, ncols=len(gia_groups), figsize=(6 * len(gia_groups), 6))

# Loop through each gia group
for i, gia in enumerate(gia_groups):
    # Subset the data for the specific gia
    subset_data = merged_df_unique_dups_removed_age_filtered_with_runtime[
        merged_df_unique_dups_removed_age_filtered_with_runtime["gia"] == gia]

    # Perform multiple regression for the current gia group
    model = sm.OLS.from_formula(formula, data=subset_data).fit()

    # Scatterplot with regression line
    sns.regplot(x='age_blooddraw', y='LENGTH_ESTIMATE', data=subset_data, ax=axes[i], 
                label=gia_groups[i], color=colors[i], 
                line_kws={'label': 'Regression Line','color': 'red'}, scatter_kws={'alpha': 0.5})
    axes[i].set_title(f'gia: {gia}')

    # Annotate plot with regression coefficient (slope)
    slope = model.params['age_blooddraw']
    axes[i].annotate(f'Slope: {slope:.6f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=12,)
    slope = model.params['age_blooddraw']
    intercept = model.params['Intercept']
    #equation = f'y = {slope:.5f} * age + {intercept:.4f}'
    #axes[i].annotate(equation, xy=(0.05, 0.85), xycoords='axes fraction', fontsize=16)
    #Annotate the regression line with the equation
    axes[i].text(0.6, 0.25, f'y = {slope:.5f} * x + {intercept:.4f}', color='black', 
        fontsize=18, transform=axes[i].transAxes, ha='center', va='center')

    # Set labels and title
    axes[i].set_xlabel("Age")
    axes[i].set_ylabel("Estimated LTL (kbp)")
    axes[i].set_title(f"GIA: {gia}")
    axes[i].legend()
    axes[i].set_ylim([0, 6])
    axes[i].set_xticks([0, 20, 40, 60, 80, 100])
    
# Adjust layout
plt.tight_layout()
fig.suptitle(f'Figure 12 : Regression plots for Telomere length ~ age + sex (each GIA)', y=1.05)
plt.show()

### 6.4. Model 4 (Cubic Spline) - LTL ~ age + sex (each ancestry) - Restricting the spline model to include ages between 20 and 80

In [ ]:
# List of gia groups
gia_groups = ["EUR", "AFR", "AMR"]

# Loop through each gia group
for gia in gia_groups:
    # Subset the data for the specific gia
    subset_data = merged_df_unique_dups_removed_age_filtered_with_runtime[
        merged_df_unique_dups_removed_age_filtered_with_runtime["gia"] == gia]

    # Filter data for the desired age range (20-100 years)
    subset_data_age_restricted = subset_data[(subset_data["age_blooddraw"] >= 20) & 
        (subset_data["age_blooddraw"] <= 100)]

    # Generate splines for age
    # Change knots to 40, 60 as majority of MVP participants were enrolled at age >=20
    knots = [40, 60] 
    x_spline_age_restricted = dmatrix("bs(age_blooddraw, knots=knots, degree=3, include_intercept=True)", 
                             {"age_blooddraw": subset_data_age_restricted["age_blooddraw"]})

    # Fit the spline model
    model_spline_age_restricted = sm.OLS(subset_data_age_restricted["LENGTH_ESTIMATE"], 
                                         x_spline_age_restricted).fit()

    # Get model summary
    model_spline_age_restricted_summary = model_spline_age_restricted.summary()

    # Print or store the results
    print(f"GIA: {gia}")
    print(model_spline_age_restricted_summary)

In [ ]:
#Telomere Length ~ age + sex (each GIA) - Regression line and superimposed fitted spline

# Define the knots for spline
knots = [40, 60]  # Adjust knots as needed

# Create a list of age values for plotting
age_values = np.linspace(20, 80, 80)  # Adjust the range as needed

# List of GIA groups and their corresponding colors
gia_groups = ["EUR", "AFR", "AMR"]
colors = ["steelblue", "darkorchid","darkorange"]

# Set up a single row of subplots
fig, axes = plt.subplots(1, len(gia_groups), figsize=(6 * len(gia_groups), 6))

# Initialize lists to collect legend handles and labels
scatter_handles = []
scatter_labels = []
spline_handle = None
regression_handle = None

# Loop through each GIA group
for i, gia in enumerate(gia_groups):
    # Subset the data for the specific GIA
    subset_data = merged_df_unique_dups_removed_age_filtered_with_runtime[
        merged_df_unique_dups_removed_age_filtered_with_runtime["gia"] == gia]

    # Filter data for the desired age range (20-100 years)
    subset_data_age_restricted = subset_data[(subset_data["age_blooddraw"] >= 20) & (subset_data["age_blooddraw"] <= 80)]

    # Generate the spline values for age
    x_spline_age_restricted = dmatrix("bs(age_blooddraw, knots=knots, degree=3, include_intercept=True)", {"age_blooddraw": age_values})

    # Fit spline model to the subset data
    x_spline_subset = dmatrix("bs(age_blooddraw, knots=knots, degree=3, include_intercept=True)", {"age_blooddraw": subset_data_age_restricted["age_blooddraw"]})
    spline_model = sm.OLS(subset_data_age_restricted["LENGTH_ESTIMATE"], x_spline_subset).fit()

    # Predict telomere length using the fitted spline model
    telomere_length_pred_age_restricted = spline_model.predict(x_spline_age_restricted)

    # Plot the observed data with the corresponding color
    scatter_handle = axes[i].scatter(subset_data_age_restricted["age_blooddraw"], subset_data_age_restricted["LENGTH_ESTIMATE"], 
                                    color=colors[i], label=f'{gia} - Data')
    scatter_handles.append(scatter_handle)
    scatter_labels.append(f'{gia} - Data')

    # Plot the fitted spline curve
    spline_handle = axes[i].plot(age_values, telomere_length_pred_age_restricted, label="Fitted Spline", color="yellow")

    # Perform linear regression on the subset data
    linear_model = sm.OLS(subset_data_age_restricted["LENGTH_ESTIMATE"], sm.add_constant(subset_data_age_restricted["age_blooddraw"])).fit()

    # Predict linear regression line
    linear_regression_line = linear_model.predict(sm.add_constant(age_values))

    # Plot the linear regression line
    regression_handle = axes[i].plot(age_values, linear_regression_line, label="Regression Line", color="black", linestyle="--")

    # Set labels and title
    axes[i].set_xlabel("Age")
    axes[i].set_ylabel("Estimated LTL(kbp)")
    axes[i].set_title(f"{gia}")
    axes[i].set_ylim([0, 6])
    axes[i].set_xticks([0, 20, 40, 60, 80, 100])

# Add a single legend to the figure
# Collect all handles and labels for the legend
handles = scatter_handles + [handle[0] for handle in [spline_handle, regression_handle]]
labels = scatter_labels + ["Fitted Spline", "Regression Line"]
fig.legend(handles=handles, labels=labels, loc='upper left', bbox_to_anchor=(1.0, 1.0), ncol=1)

# Adjust layout and show the plots
plt.tight_layout() 
fig.suptitle(f'Figure 13 : Telomere Length ~ age + sex (each GIA) - Regression line and fitted spline', y=1.05)
plt.show()


## 6. LLR test

In [ ]:

# List of ancestry groups
ancestry_groups = ["EUR", "AFR", "AMR"]

# Initialize dictionaries to store log-likelihoods and likelihood ratio test results
log_likelihoods_linear = {}
log_likelihoods_spline = {}
likelihood_ratio_test_results = {}

# Loop through each ancestry group
for ancestry in ancestry_groups:
    # Subset the data for the specific ancestry
    subset_data = merged_df_unique_dups_removed_age_filtered_with_runtime[
        merged_df_unique_dups_removed_age_filtered_with_runtime["gia"] == ancestry]

    # Model 3: Linear regression - Telomere length ~ age + sex (each ancestry)
    formula = 'LENGTH_ESTIMATE ~ age_blooddraw + sex'
    model_linear = sm.OLS.from_formula(formula, data=subset_data).fit()
    log_likelihood_linear = model_linear.llf  # Extract log-likelihood
    log_likelihoods_linear[ancestry] = log_likelihood_linear

    # Model 4: Spline model - Telomere length ~ age + sex (each ancestry)
    knots = [40, 60, 80]
    x_spline = dmatrix("bs(age_blooddraw, knots=knots, degree=3, include_intercept=True)", 
                        {"age_blooddraw": subset_data["age_blooddraw"]})
    model_spline = sm.OLS(subset_data["LENGTH_ESTIMATE"], x_spline).fit()
    log_likelihood_spline = model_spline.llf  # Extract log-likelihood
    log_likelihoods_spline[ancestry] = log_likelihood_spline

    # Calculate likelihood ratio test statistic
    lr_test_statistic = 2 * (log_likelihood_spline - log_likelihood_linear)

    # Calculate degrees of freedom (difference in number of parameters between models)
    df = model_spline.df_model - model_linear.df_model

    # Calculate p-value using chi-square distribution
    p_value = 1 - chi2.cdf(lr_test_statistic, df)

    # Store likelihood ratio test results
    likelihood_ratio_test_results[ancestry] = {'LR Statistic': lr_test_statistic, 'Degrees of Freedom': df, 'P-value': p_value}

# Print likelihood ratio test results for each ancestry
for ancestry, result in likelihood_ratio_test_results.items():
    print(f"Ancestry: {ancestry}")
    print("Likelihood Ratio Test Statistic:", result['LR Statistic'])
    print("Degrees of Freedom:", result['Degrees of Freedom'])
    print("P-value:", result['P-value'])
    print()


In [ ]:
# Perform ANOVA comparing the two models
anova_llr_results = sm.stats.anova_lm(model_linear, model_spline, typ=1)

# Print ANOVA results
print("ANOVA Results Comparing Linear and Spline Models:")
print(anova_llr_results)

## 7. Mosdepth - NCG PCA pipeline for 102k samples 

### 7.1. Data loading - Results

In [ ]:
blob = bucket.blob("PATH/TO/results_ngspca_102ksvd.pcs.txt")

# Download blob content into memory as bytes
data_bytes = blob.download_as_bytes()

# Read into pandas directly
import io
mosdepth_102k_pcs = pd.read_csv(io.BytesIO(data_bytes), sep='\t')

#Rename SAMPLE column to ship-id 
mosdepth_102k_pcs.rename(columns={'SAMPLE':'ship-id'}, inplace=True)

# Remove the . after ship-id
mosdepth_102k_pcs['ship-id'] = (mosdepth_102k_pcs['ship-id'].astype(str).str.replace(r'.$', '', regex=True))


# Optional: preview the DataFrame
print(len(mosdepth_102k_pcs))
mosdepth_102k_pcs.head()

### 7.2. Data loading - PC and singular values for samples from GCS

In [ ]:
blob = bucket.blob("PATH/TO/results_ngspca_102ksvd.singularvalues.txt")

# Download blob content into memory as bytes
data_bytes = blob.download_as_bytes()

# Read into pandas directly
import io
mosdepth_102k_sing_vals = pd.read_csv(io.BytesIO(data_bytes), sep='\t')

#Rename SAMPLE column to ship-id 
mosdepth_102k_sing_vals.rename(columns={'SAMPLE':'ship-id'}, inplace=True)

# Remove the . after ship-id
#mosdepth_102k_sing_vals['ship-id'] = (mosdepth_102k_sing_vals['ship-id'].astype(str).str.replace(r'.$', '', regex=True))


# Optional: preview the DataFrame
print(len(mosdepth_102k_sing_vals))
mosdepth_102k_sing_vals.head()

### 7.3. Merge LTL data with 200 PCs

In [ ]:
pc_cols = [f'PC{i}' for i in range(1, 201)]

mosdepth_res_with_ltl = pd.merge(merged_df_unique_dups_removed_age_filtered_with_runtime[['ship-id', 'age_blooddraw', 'gia', 'LENGTH_ESTIMATE', 'sex', 'phenotype']],
                                         mosdepth_102k_pcs[['ship-id'] + pc_cols],
                                         on='ship-id', how='left')

print(len(mosdepth_res_with_ltl))
mosdepth_res_with_ltl.head()

In [ ]:
# Calculate cumulative variance
mosdepth_102k_sing_vals['CUMULATIVE_VAR'] = mosdepth_102k_sing_vals['PERCENT_VAR'].cumsum()

# Find the first PC index where cumulative variance exceeds 80%
threshold_80 = mosdepth_102k_sing_vals[mosdepth_102k_sing_vals['CUMULATIVE_VAR'] >= 80].iloc[0]

print(f"PC1 through PC{int(threshold_80['PC'])} explains {threshold_80['CUMULATIVE_VAR']:.2f}% of the variance.")

### 16.4.8. Filtering for GIA (AFR, EUR and AMR) and <=80 age

### 7.5. Regression models

#### 7.5.1. LTL ~ Age + Sex + Ancestry (BEFORE regressing out the PCs)

In [ ]:
m1_ltl = ols(formula='LENGTH_ESTIMATE ~ age_blooddraw + sex', data=mosdepth_res_with_ltl).fit()
r1_ltl = m1_ltl.summary()
print(r1_ltl)

#### 7.5.2. LTL ~ age + sex (AFTER regressing out the PCs)

In [ ]:
# 1. Including the 200PCs as covariates in the formula
pc_cols = [f'PC{i}' for i in range(1, 201)]

# 2. Formula string for the regression model with PCs
formula_str = 'LENGTH_ESTIMATE ~ age_blooddraw + sex + ' + ' + '.join(pc_cols)

# 3. Fit the model
m1_ltl_pcs = ols(formula=formula_str, data=mosdepth_res_with_ltl).fit()

r1_ltl_pcs = m1_ltl_pcs.summary()
print(r1_ltl_pcs)

#### 7.5.3. LTL ~ Age + Sex + Ancestry

In [ ]:
# Specify 'gia' as categorical and set 'gia_EUR' as the reference category
m2_102k_ltl = ols(formula='LENGTH_ESTIMATE ~ age_blooddraw + sex + C(gia, Treatment("EUR"))', data=mosdepth_res_with_ltl).fit()
results2_102k_ltl = m2_102k_ltl.summary()
print(results2_102k_ltl)

#### 7.5.4. Residualized LTL ~ Age + Sex + GIA (Ancestry)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Define the covariates for residualization
pc_cols = [f'PC{i}' for i in range(1, 201)]
df_clean = mosdepth_res_with_ltl.dropna(
    subset=['LENGTH_ESTIMATE', 'age_blooddraw', 'sex', 'gia'] + pc_cols
).copy()

# STEP 1: Create Residualized LTL (LTL ~ 200 PCs)
# We use the residuals to represent LTL stripped of technical/batch effects
pc_formula = "LENGTH_ESTIMATE ~ " + " + ".join(pc_cols)
temp_model = smf.ols(pc_formula, data=df_clean).fit()
df_clean['LTL_resids'] = temp_model.resid

# STEP 2: Run Model M2 (The Biological Model)
# LTL_resids ~ Age + Sex + Ancestry (GIA)
# We use C() for categorical variables like Sex and GIA
m2_formula = "LTL_resids ~ age_blooddraw + C(sex) + C(gia, Treatment('EUR'))"
m2_model = smf.ols(m2_formula, data=df_clean).fit()

# Print the Results
print(m2_model.summary())

#### 7.5.5 LTL ~ Age + Sex + Ancestry + NGS-PC1-200 

In [ ]:
# 1. Including the first 20 PCs as covariates in the formula
pc_cols = [f'PC{i}' for i in range(1, 201)]

# 2. Build the formula string
# We join the PCs with ' + ' and add them to your base variables
pc_formula_part = ' + '.join(pc_cols)
base_formula = 'LENGTH_ESTIMATE ~ age_blooddraw + sex + C(gia, Treatment("EUR"))'
full_formula = f"{base_formula} + {pc_formula_part}"

# 3. Fit the model using the expanded formula
m3_102k_ltl_pcs = ols(formula=full_formula, data=mosdepth_res_with_ltl).fit()

# 4. View results
r3_102k_ltl_pcs = m3_102k_ltl_pcs.summary()
print(r3_102k_ltl_pcs)

### 7.6. Descriptive statistics

In [ ]:
#Male
male_sample_ltl_pcs = mosdepth_res_with_ltl[
    mosdepth_res_with_ltl['sex'] == 'M']

#Female
female_sample_ltl_pcs = mosdepth_res_with_ltl[
    mosdepth_res_with_ltl['sex'] == 'F']

print("Number of samples - by Sex") 
print("Male = ",len(male_sample_ltl_pcs)) 
print("Female = ",len(female_sample_ltl_pcs))  


#### 7.6.1. Descriptive statistics of Age by Sex 

In [ ]:
# Define a dictionary with ancestry groups and their respective sample dataframes
sex_groups = {
    "Male": male_sample_ltl_pcs,
    "Female": female_sample_ltl_pcs,
}
# Initialize lists to store mean ages and standard deviations
mean_age_sex_102k = []
std_dev_age_sex_102k = []

# Loop over each ancestry group
for group, sample_df in sex_groups.items():
    # Calculate the mean age for the ancestry group
    mean_age_sex_df = sample_df['age_blooddraw'].mean()
    # Calculate the standard deviation age for the ancestry group
    std_dev_age_sex_df = sample_df['age_blooddraw'].std()
    # Append the results to the lists
    mean_age_sex_102k.append(mean_age_sex_df)
    std_dev_age_sex_102k.append(std_dev_age_sex_df)

# Create a DataFrame to store the results
results_df_age_sex_102k = pd.DataFrame({'N': [len(sample_df) for sample_df in sex_groups.values()], 'Mean Age(kbp)': mean_age_sex_102k, 'SD(kbp)': std_dev_age_sex_102k}, index=sex_groups.keys())

results_df_age_sex_102k.sort_values(by='N', inplace=True, ascending=False)
print(results_df_age_sex_102k)

#### 7.6.2. Descriptive statistics of LTL by Sex

In [ ]:
# Define a dictionary with ancestry groups and their respective sample dataframes
sex_groups = {
    "Male": male_sample_ltl_pcs,
    "Female": female_sample_ltl_pcs,
}
# Initialize lists to store mean ages and standard deviations
mean_ltl_sex_102k = []
std_dev_ltl_sex_102k = []

# Loop over each ancestry group
for group, sample_df in sex_groups.items():
    # Calculate the mean age for the ancestry group
    mean_ltl_sex_df = sample_df['LENGTH_ESTIMATE'].mean()
    # Calculate the standard deviation age for the ancestry group
    std_dev_ltl_sex_df = sample_df['LENGTH_ESTIMATE'].std()
    # Append the results to the lists
    mean_ltl_sex_102k.append(mean_ltl_sex_df)
    std_dev_ltl_sex_102k.append(std_dev_ltl_sex_df)

# Create a DataFrame to store the results
results_df_ltl_sex_102k = pd.DataFrame({'N': [len(sample_df) for sample_df in sex_groups.values()], 'Mean LTL(kbp)': mean_ltl_sex_102k, 'SD(kbp)': std_dev_ltl_sex_102k}, index=sex_groups.keys())

results_df_ltl_sex_102k.sort_values(by='N', inplace=True, ascending=False)
print(results_df_ltl_sex_102k)

#### 7.6.3. Descriptive statistics of PC adjusted LTL by Sex 

In [ ]:

# 1. Define PC columns and prepare dataset
pc_cols = [f'PC{i}' for i in range(1, 201)]
df_adj_102k = mosdepth_res_with_ltl.dropna(subset=['LENGTH_ESTIMATE'] + pc_cols).copy()

# 2. Regress out the 200 PCs from LTL (residuals + original mean to preserve kbp scale)
model_ltl = sm.OLS(df_adj_102k['LENGTH_ESTIMATE'], sm.add_constant(df_adj_102k[pc_cols])).fit()
df_adj_102k['LTL_ADJUSTED'] = model_ltl.resid + df_adj_102k['LENGTH_ESTIMATE'].mean()

# 3. Filter/assign dataset into ancestry groups
sex_groups = {
    "Male": df_adj_102k[df_adj_102k['sex'] == 'M'],
    "Female": df_adj_102k[df_adj_102k['sex'] == 'F'],
}

# 4. Initialize lists to store metrics
sample_sizes_sex_102k = []
mean_ltl_adj_sex_102k = []
std_dev_ltl_adj_sex_102k = []

# 5. Calculate metrics for each ancestry group using 'LTL_ADJUSTED'
for group, sample_df in sex_groups.items():
    sample_sizes_sex_102k.append(len(sample_df))
    mean_ltl_adj_sex_102k.append(sample_df['LTL_ADJUSTED'].mean())
    std_dev_ltl_adj_sex_102k.append(sample_df['LTL_ADJUSTED'].std())

# 6. Create and format the adjusted results DataFrame
results_df_ltl_sex_adj_102k = pd.DataFrame(
    {
        'N': sample_sizes_sex_102k, 
        'Mean Adj LTL(kbp)': mean_ltl_adj_sex_102k, 
        'Adj SD(kbp)': std_dev_ltl_adj_sex_102k
    }, 
    index=sex_groups.keys()
)

results_df_ltl_sex_adj_102k.sort_values(by='N', ascending=False, inplace=True)

print(results_df_ltl_sex_adj_102k)

            N  Mean Adj LTL(kbp)  Adj SD(kbp)
Male    79547           2.041387     0.256951
Female   5733           2.174227     0.287281


### 7.7. Descriptive statistics by GIA

In [ ]:
#EUR
eur_sample_ltl_pcs = mosdepth_res_with_ltl[
    mosdepth_res_with_ltl['gia'] == 'EUR']

#AFR
afr_sample_ltl_pcs = mosdepth_res_with_ltl[
    mosdepth_res_with_ltl['gia'] == 'AFR']

#AMR
amr_sample_ltl_pcs = mosdepth_res_with_ltl[
    mosdepth_res_with_ltl['gia'] == 'AMR']

print("Number of samples per GIA group") 
print("European = ",len(eur_sample_ltl_pcs)) 
print("African = ",len(afr_sample_ltl_pcs))  
print("American = ",len(amr_sample_ltl_pcs)) 

Number of samples per GIA group
European =  56928
African =  22628
American =  5741


#### 7.7.1. Descriptive statistics of Age by GIA

In [ ]:
# Define a dictionary with ancestry groups and their respective sample dataframes
ancestry_groups = {
    "European": eur_sample_ltl_pcs,
    "African": afr_sample_ltl_pcs,
    "American": amr_sample_ltl_pcs
}
# Initialize lists to store mean ages and standard deviations
mean_age_gia_102k = []
std_dev_age_gia_102k = []

# Loop over each ancestry group
for group, sample_df in ancestry_groups.items():
    # Calculate the mean age for the ancestry group
    mean_age_gia_df = sample_df['age_blooddraw'].mean()
    # Calculate the standard deviation age for the ancestry group
    std_dev_age_gia_df = sample_df['age_blooddraw'].std()
    # Append the results to the lists
    mean_age_gia_102k.append(mean_age_gia_df)
    std_dev_age_gia_102k.append(std_dev_age_gia_df)

# Create a DataFrame to store the results
results_df_age_gia_102k = pd.DataFrame({'N': [len(sample_df) for sample_df in ancestry_groups.values()], 'Mean(years)': mean_age_gia_102k, 'SD(years)': std_dev_age_gia_102k}, index=ancestry_groups.keys())

results_df_age_gia_102k.sort_values(by='N', inplace=True, ascending=False)
print(results_df_age_gia_102k)

#### 7.7.2. Descriptive statistics of LTL by GIA 

In [ ]:
# Define a dictionary with ancestry groups and their respective sample dataframes
ancestry_groups = {
    "European": eur_sample_ltl_pcs,
    "African": afr_sample_ltl_pcs,
    "American": amr_sample_ltl_pcs
}
# Initialize lists to store mean ages and standard deviations
mean_ltl_gia_102k = []
std_dev_ltl_gia_102k = []

# Loop over each ancestry group
for group, sample_df in ancestry_groups.items():
    # Calculate the mean age for the ancestry group
    mean_ltl_gia_df = sample_df['LENGTH_ESTIMATE'].mean()
    # Calculate the standard deviation age for the ancestry group
    std_dev_ltl_gia_df = sample_df['LENGTH_ESTIMATE'].std()
    # Append the results to the lists
    mean_ltl_gia_102k.append(mean_ltl_gia_df)
    std_dev_ltl_gia_102k.append(std_dev_ltl_gia_df)

# Create a DataFrame to store the results
results_df_ltl_gia_102k = pd.DataFrame({'N': [len(sample_df) for sample_df in ancestry_groups.values()], 'Mean LTL(kbp)': mean_ltl_gia_102k, 'SD(kbp)': std_dev_ltl_gia_102k}, index=ancestry_groups.keys())

results_df_ltl_gia_102k.sort_values(by='N', inplace=True, ascending=False)
print(results_df_ltl_gia_102k)

#### 7.7.2. Descriptive statistics of PC Adjusted LTL by GIA 

In [ ]:
import pandas as pd
import statsmodels.api as sm

# 1. Define PC columns and prepare dataset
pc_cols = [f'PC{i}' for i in range(1, 201)]
df_adj_102k = mosdepth_res_with_ltl.dropna(subset=['LENGTH_ESTIMATE'] + pc_cols).copy()

# 2. Regress out the 200 PCs from LTL (residuals + original mean to preserve kbp scale)
model_ltl = sm.OLS(df_adj_102k['LENGTH_ESTIMATE'], sm.add_constant(df_adj_102k[pc_cols])).fit()
df_adj_102k['LTL_ADJUSTED'] = model_ltl.resid + df_adj_102k['LENGTH_ESTIMATE'].mean()

# 3. Filter/assign dataset into ancestry groups
ancestry_groups = {
    "European": df_adj_102k[df_adj_102k['gia'] == 'EUR'],
    "African": df_adj_102k[df_adj_102k['gia'] == 'AFR'],
    "American": df_adj_102k[df_adj_102k['gia'] == 'AMR']
}

# 4. Initialize lists to store metrics
sample_sizes_102k = []
mean_ltl_adj_102k = []
std_dev_ltl_adj_102k = []

# 5. Calculate metrics for each ancestry group using 'LTL_ADJUSTED'
for group, sample_df in ancestry_groups.items():
    sample_sizes_102k.append(len(sample_df))
    mean_ltl_adj_102k.append(sample_df['LTL_ADJUSTED'].mean())
    std_dev_ltl_adj_102k.append(sample_df['LTL_ADJUSTED'].std())

# 6. Create and format the adjusted results DataFrame
results_df_ltl_gia_adj_102k = pd.DataFrame(
    {
        'N': sample_sizes_102k, 
        'Mean Adj LTL(kbp)': mean_ltl_adj_102k, 
        'Adj SD(kbp)': std_dev_ltl_adj_102k
    }, 
    index=ancestry_groups.keys()
)

results_df_ltl_gia_adj_102k.sort_values(by='N', ascending=False, inplace=True)

print(results_df_ltl_gia_adj_102k)

### 8. Idenitfying missing PCS within GIA groups

In [ ]:

# Check missing values per ancestry group before dropping them
missing_by_group = mosdepth_res_with_ltl.groupby('gia')[pc_cols].apply(lambda x: x.isna().sum().sum())

print("Total missing PC entries by group:")
print(missing_by_group)

In [ ]:
# Count how many individual rows have ANY missing PC values
missing_rows_by_group = (
    mosdepth_res_with_ltl[pc_cols]
    .isna()
    .any(axis=1)
    .groupby(
        mosdepth_res_with_ltl['gia']
    )
    .sum()
)

print('Number of dropped sample rows per ancestry group:')
print(missing_rows_by_group)